# 街景字符识别 - FPN Multi-Head 模型

基于 PyTorch 的街景字符识别项目，采用 FPN + Multi-Head Attention 架构。

**支持三种模型**：`fpn_multihead`（默认）、`transformer`、`ctc`

**支持两种GPU环境**：
- NVIDIA A100 (8核 / 32GB内存 / 24GB显存)
- AMD ROCm (8核 / 200GB内存 / 192GB显存)

运行时自动检测GPU环境并选择对应的参数配置（GPUProfile策略模式）。

**torch.compile 支持**：AMD ROCm 环境下自动启用 `torch.compile`，首次运行需编译 kernel（约5-15分钟），后续运行使用缓存。

**日志路径**：`{项目目录}/logs/train_YYYYMMDD_HHMMSS.log`

In [ ]:
# 环境配置：设置随机种子、检测GPU、打印版本信息
import torch as t
import sys
import os

from utils.seed import set_seed
from config import config, BASE_DIR, data_dir, IS_MODELSCOPE, GPU_PLATFORM, TOTAL_VRAM_GB, IS_NVIDIA, IS_AMD
from utils.platform import get_precision_config, is_nvidia_cuda

# 平台特定精度配置
precision_config = get_precision_config()
if is_nvidia_cuda():
    try:
        t.set_float32_matmul_precision('high')
        print('TF32 matmul precision: enabled')
    except Exception:
        print('TF32 matmul precision: not supported on this GPU')
    try:
        t.backends.cudnn.allow_tf32 = True
        print('CuDNN TF32: enabled')
    except Exception:
        print('CuDNN TF32: not supported')
else:
    print('TF32/CuDNN: platform-specific settings skipped')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

set_seed(42)

print('=' * 60)
print('环境信息')
print('=' * 60)
print(f'Python: {sys.version}')
print(f'PyTorch: {t.__version__}')
print(f'GPU Platform: {GPU_PLATFORM.upper()}')
print(f'CUDA available: {t.cuda.is_available()}')
if t.cuda.is_available():
    print(f'CUDA version: {t.version.cuda}')
    print(f'GPU: {t.cuda.get_device_name(0)}')
    props = t.cuda.get_device_properties(0)
    vram = getattr(props, 'total_mem', getattr(props, 'total_memory', 0)) / (1024**3)
    print(f'GPU VRAM: {vram:.1f} GB')
    try:
        t.set_float32_matmul_precision('high')
        print('TF32 matmul precision: enabled')
    except Exception:
        print('TF32 matmul precision: not supported on this GPU')
print(f'BASE_DIR: {BASE_DIR}')
print(f'ModelScope环境: {IS_MODELSCOPE}')
print(f'Model type: {config.model_type}')
print(f'Batch size: {config.batch_size}')
print(f'Eval batch size: {config.eval_batch_size}')
print(f'Num workers: {config.num_workers}')
print(f'Gradient accumulation: {config.grad_accum_steps}')
print(f'Equivalent batch: {config.batch_size * config.grad_accum_steps}')
print(f'Input size: {config.input_height}x{config.input_width}')
print(f'Use torch.compile: {config.use_torch_compile}')
if config.use_torch_compile:
    print(f'Compile mode: {config.compile_mode}')
    print(f'Compile dynamic: {config.compile_dynamic}')
    print(f'Compile fullgraph: {config.compile_fullgraph}')
print(f'Gradient checkpoint: {config.use_gradient_checkpoint}')
print('=' * 60)

In [ ]:
# 下载数据集（仅在数据不存在时执行）
from data.download import download_dataset
download_dataset()

## 数据探索

In [ ]:
# 统计训练集、验证集、测试集的图片数量
from glob import glob

train_list = glob(data_dir['train_data'] + '*.png')
val_list = glob(data_dir['val_data'] + '*.png')
test_list = glob(data_dir['test_data'] + '*.png')

print(f'训练集图片数: {len(train_list)}')
print(f'验证集图片数: {len(val_list)}')
print(f'测试集图片数: {len(test_list)}')

In [ ]:
# 分析训练集中数字位数的分布
import json

with open(data_dir['train_label'], 'r') as f:
    marks = json.load(f)

dicts = {}
for img, mark in marks.items():
    n = len(mark['label'])
    dicts[n] = dicts.get(n, 0) + 1
for k, v in sorted(dicts.items()):
    print(f'{k}位数字的图片数目: {v}')

## 模型定义

创建模型并统计参数量

In [ ]:
# 创建模型并统计参数量
from models import create_model

model = create_model('fpn_multihead')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型类型: fpn_multihead')
print(f'总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')
print(f'模型大小: {total_params * 4 / (1024**2):.1f} MB (FP32)')

## 训练 FPN Multi-Head 模型

训练过程中日志会自动记录到 `logs/` 目录下，包含：
- 每个batch的loss、学习率、精度、GPU/CPU内存使用
- 每个epoch的训练/验证精度（joint acc + char acc）、耗时、early stopping状态

**torch.compile warmup 说明**：
- AMD ROCm 环境下自动启用 `torch.compile(mode='default', dynamic=True)`
- 首次运行需编译 CUDA kernel，warmup 阶段约需 **5-15 分钟**（取决于缓存状态）
- 后续运行使用 inductor_cache 缓存，warmup 时间大幅缩短
- 编译失败时自动 fallback 到 eager 模式，不影响训练

In [ ]:
# 训练FPN Multi-Head模型
# 日志自动保存到 {项目目录}/logs/ 目录
from trainer.multihead import MultiHeadTrainer
from utils.misc import find_latest_checkpoint

latest_checkpoint = find_latest_checkpoint(config.checkpoints)
if latest_checkpoint:
    print(f'发现检查点: {latest_checkpoint}')
    config.pretrained = latest_checkpoint

trainer = MultiHeadTrainer(model_type='fpn_multihead')
trainer.train()

## 评估

In [ ]:
# 在验证集上评估模型精度（返回joint acc）
val_acc = trainer._eval()
print(f'最佳验证精度: {trainer.best_acc * 100:.2f}%')

In [ ]:
# 逐Head评估每个数字位的分类精度
trainer.eval_detailed()

## torch.compile 审计（可选）

运行系统性编译审计，覆盖编译模式、输入数据类型/尺寸/批次、warmup策略对比、正确性验证等。
完整审计约需 1-2 小时，可按需选择单项审计。

In [ ]:
# 运行 torch.compile 系统性审计（可选，取消注释执行）
# !python scripts/compile_audit.py --audit warmup  # 仅测试warmup策略
# !python scripts/compile_audit.py --audit modes   # 仅测试编译模式
# !python scripts/compile_audit.py --audit all     # 完整审计

## 推理与提交

In [ ]:
# 推理（use_compile=True 时启用 torch.compile 加速推理）
from inference.predict import predicts, ctc_predict, ensemble_predict

# model_path = 'checkpoints/best-resnet101-acc-xx.xx.pth'
# results = predicts(model_path, 'submit.csv', use_tta=True, use_compile=True)

In [ ]:
# TTA 评估
# tta_acc = trainer.eval_tta()
# print(f'TTA Accuracy: {tta_acc * 100:.2f}%')

## 训练日志

In [ ]:
# 查看训练日志
for entry in trainer.train_log[-5:]:
    print(entry)

In [ ]:
# 保存模型
# trainer.save_model('checkpoints/manual_save.pth', save_opt=True)